> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — fine mapping, LD and SHAP-ready locus representation

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
# Genotypes are simulated once into a 2-bit packed Zarr store (GT_packed2, 0.25 byte per call)
# that every tutorial reuses. Traits are simulated from bounded decoded blocks, so no dense
# animals x variants matrix is held in memory; GENO is the genotype source used below.
DATA=Path("vigor_tutorial_data"); DATA.mkdir(exist_ok=True)
GENO=str(DATA/"complex_n320_m1000_chr5.zarr")
sim=vigor.simulate_packed_cohort(GENO,n_samples=320,n_variants=1000,n_chromosomes=5,n_traits=8,seed=SEED,genotype_kwargs={"generations":100},trait_kwargs=cfg)
Y=sim.phenotypes.to_pandas(); VI=vigor.result_table(sim,"variant_info")
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False

# Fine mapping needs a dense region: decode one bounded window of the packed store.
region=vigor.genotype_panel(GENO,chromosome=str(VI.chrom.iloc[0]),variants=slice(0,120))
ld=vigor.compute_variant_ld(region,statistic="both")
# `compute_variant_ld` returns the LD objects used by clustering/fine mapping.
print(type(ld))
# Small ridge/elastic-net fine-map smoke test on one trait.
fm=vigor.fine_map_region(region,Y.iloc[:,[0]],methods=("ridge","elasticnet"))
print(type(fm))


TreeSHAP explains the nonlinear expert but is not itself LD-aware; aggregate out-of-fold SHAP to LD blocks for locus-level interpretation.

In [ ]:
fm_shap=vigor.fine_map_region(region,train.iloc[:,:1],methods=("ridge",),compute_linear_shap=True)
print("Fine-map result with SHAP:",type(fm_shap).__name__)


## Multi-trait Bayesian fine mapping, GTF gene priors and PCSF export

`methods=("bayes",)` runs the native multi-trait IBSS fine-mapper. By default
(`residual_covariance='fit'`) it estimates the full residual trait covariance by
variational ECM, including posterior effect uncertainty; `'diagonal'` restores the
older fixed-diagonal behaviour. Each single effect reports its evidence and LD purity
in `component_diagnostics`. Only effects passing both screens form
`selected_credible_sets`. Add `bayes_kwargs={"null_permutations": 50}` for an
exchangeable global-null permutation p-value.

A GTF turns positions into **positional priors** (CDS > gene body > intergenic) and
uncalibrated per-gene PIP summaries. Neither is a causal-gene posterior.
`pcsf_export=True` writes allele-oriented posterior betas, SNP→gene edges and PIP
prizes for the PCSF solver used in tutorial 10.

In [ ]:
import gzip
# Pick a window around the strongest planted additive effect of the first three traits. Truth
# only chooses where to look; the fit below never sees it.
traits3=list(Y.columns[:3])
VE=vigor.result_table(sim,"variant_effects")
VE=VE[VE.trait.isin(traits3)]
lead=str(VE.loc[VE.additive_effect_final_mean.abs().idxmax(),"snp"])
chrom=str(VI.loc[lead,"chrom"]); on_chrom=VI.index[VI.chrom.astype(str)==chrom]
k=int(on_chrom.get_loc(lead))
Xw=vigor.genotype_panel(GENO,chromosome=chrom,variants=slice(max(0,k-40),k+40))  # bounded dense window
# The packed store has no allele labels: A1/A2 below are synthetic, with dosage counting A2.
vw=pd.DataFrame({"chrom":chrom,"pos":VI.loc[Xw.columns,"pos"].astype(int).to_numpy(),"A1":"A","A2":"G"},index=Xw.columns)
# Synthetic GTF: one 4-variant gene every 8 variants, tiled independently of the simulated truth,
# with a CDS over each gene's first variant. Real analyses use the assembly-matched RefSeq/Ensembl
# GTF (gtf_contig_aliases maps "12" to "NC_086030.1", annotation_assembly/variant_assembly check it).
gtf=DATA/f"synthetic_chr{chrom}_window.gtf.gz"
with gzip.open(gtf,"wt") as f:
    for g,i in enumerate(range(0,len(vw)-3,8)):
        a,b=max(1,int(vw.pos.iloc[i])-50),int(vw.pos.iloc[i+3])+50
        f.write(f'{chrom}\tsynthetic\tgene\t{a}\t{b}\t.\t+\t.\tgene_id "SG{g}"; gene "Gene{g}"; gene_biotype "protein_coding";\n')
        f.write(f'{chrom}\tsynthetic\tCDS\t{a}\t{a+200}\t.\t+\t0\tgene_id "SG{g}"; gene "Gene{g}";\n')
bfm=vigor.fine_map_region(Xw,Y,trait_subset=traits3,methods=("bayes",),L=4,hdbscan=False,
    use_evidence_priors=False,gtf_path=gtf,gtf_variant_metadata=vw,allow_unverified_assembly=True,
    pcsf_export=True,pcsf_genotype_dosage_allele="A2",pcsf_link_cost=0.05,
    bayes_kwargs={"max_iter":100,"tol":1e-5})
bayes=bfm.fits["bayes"]
print("window",chrom,Xw.shape,"planted lead",lead)
print("residual covariance mode:",bayes["residual_covariance_mode"])
print(bayes["residual_covariance"].round(3))
print(bayes["component_diagnostics"])
print(bayes["selected_credible_sets"])
pip=bayes["pip"].set_index("snp").pip
print("PIP of the planted lead:",float(pip.get(lead,np.nan)),"| top PIPs:"); print(pip.sort_values(ascending=False).head())
print(bfm.metadata["gene_scores"].head())  # uncalibrated per-gene PIP summaries
export=bfm.metadata["pcsf_inputs"]
print(export["variant_nodes"].filter(regex="^(snp|pip|effect_allele|beta_per_A2)").head())
print(export["variant_gene_edges"].head())
